# Historic Candle Feature Builder

Use one `tradingbot.core.Features` contract for both single-candle inference and offline dataset feature files. The same builder that can transform one live `Candle` is used by the Dask file-wise export into `../data/historic_candle_feature`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PACKAGE_SRC = PROJECT_ROOT / "packages" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

PROJECT_ROOT

PosixPath('/Users/akash/PycharmProjects/AlgoTrade')

In [2]:
from tradingbot.core.features import CandleFeatureBuilder

INPUT_DIR = PROJECT_ROOT / "data" / "historic_data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "historic_candle_feature"
FILE_PATTERN = "*.csv"
OVERWRITE_EXISTING = True

FEATURES = [
    "timestamp",
    "upper_wick_to_candle_ratio",
    "lower_wick_to_candle_ratio",
    "body_to_candle_ratio",
    "candle_type",
    "candle_color_code",
    "log_volume_zscore",
]

feature_builder = CandleFeatureBuilder(FEATURES)
feature_builder.feature_map


{'timestamp': 0,
 'upper_wick_to_candle_ratio': 1,
 'lower_wick_to_candle_ratio': 2,
 'body_to_candle_ratio': 3,
 'candle_type_standard': 4,
 'candle_type_doji': 5,
 'candle_type_hammer': 6,
 'candle_type_inverted_hammer': 7,
 'candle_type_spinning_top': 8,
 'candle_type_marubozu': 9,
 'candle_color_green': 10,
 'candle_color_red': 11,
 'log_volume_zscore': 12}

In [3]:
sample_candle = {
    "timestamp": 1700000000000,
    "open": 100.0,
    "high": 110.0,      
    "low": 90.0,
    "close": 105.0,
    "volume": 1000.0
}
features = feature_builder.encode(sample_candle)
features

{'timestamp': 1700000000000,
 'upper_wick_to_candle_ratio': -0.25,
 'lower_wick_to_candle_ratio': 0.0,
 'body_to_candle_ratio': -0.25,
 'candle_type_standard': 0,
 'candle_type_doji': 0,
 'candle_type_hammer': 1,
 'candle_type_inverted_hammer': 0,
 'candle_type_spinning_top': 0,
 'candle_type_marubozu': 0,
 'candle_color_green': 1,
 'candle_color_red': 0,
 'log_volume_zscore': None}

In [4]:
import glob
import pandas as pd
from dask import dataframe as dd

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_meta = pd.DataFrame({
    column: pd.Series(dtype="object" if column == "timestamp" else "float64")
    for column in feature_builder.output_columns
})

csv_files = glob.glob(str(INPUT_DIR / FILE_PATTERN))
for csv_file in csv_files:
    output_file = OUTPUT_DIR / Path(csv_file).name
    if output_file.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {csv_file} as output already exists.")
        continue
    
    print(f"Processing {csv_file}...")
    df = dd.read_csv(csv_file, blocksize=None)
    features_df = df.map_partitions(
        feature_builder.encode_dataframe,
        meta=feature_meta,
    )
    features_df.to_csv(output_file, single_file=True, index=False)
    print(f"Saved features to {output_file}")


Processing /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/TCS_minute_1990-01-01T00-00-00plus05-30_now.csv...
Saved features to /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_feature/TCS_minute_1990-01-01T00-00-00plus05-30_now.csv
Processing /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/KOTAKBANK_minute_1990-01-01T00-00-00plus05-30_now.csv...
Saved features to /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_feature/KOTAKBANK_minute_1990-01-01T00-00-00plus05-30_now.csv
Processing /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/ICICIBANK_5minute_1990-01-01T00-00-00plus05-30_now.csv...
Saved features to /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_feature/ICICIBANK_5minute_1990-01-01T00-00-00plus05-30_now.csv
Processing /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/ICICIBANK_60minute_1990-01-01T00-00-00plus05-30_now.csv...
Saved features to /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_featur